# `hotel_booking_demand`: workbench

The definition lives in [`dataset.py`](dataset.py) and the evidence in [`README.md`](README.md). Edit `dataset.py`
and re-run a cell: `ds` picks up the change (the raw files are not read again unless `_load_raw` changed).

In [ ]:
from data_foundry.v2 import workbench

ds = workbench()
ds.df.head()

In [ ]:
ds.run_all_checks()["summary"]

In [ ]:
ds.preview_splits()

## Exploration

Migrated from the v1 notebook; adapt the cells where needed.

In [ ]:
df = ds.df.copy()  # the migrated cells below work on `df`

In [ ]:
df[["booking_date", "arrival_date", "ReservationStatusDate"]].describe()

In [ ]:
# There are seasonal patterns, but also a long-term trend towards more cancellations.
mean_per_month = df["IsCanceled"].groupby(pd.to_datetime(df[["ArrivalDateYear","ArrivalDateMonth"]].astype(str).agg('-'.join, axis=1) + '-01')).mean()
plt.scatter(mean_per_month.index,mean_per_month.values)

In [ ]:
# There are seasonal patterns, but also a long-term trend towards more cancellations.
df_resort = df[df["hotel"] == "resort"]
mean_per_month = df_resort["IsCanceled"].groupby(pd.to_datetime(df_resort[["ArrivalDateYear","ArrivalDateMonth"]].astype(str).agg('-'.join, axis=1) + '-01')).mean()
plt.scatter(mean_per_month.index,mean_per_month.values)

In [ ]:
# There are seasonal patterns, but also a long-term trend towards more cancellations.
df_city = df[df["hotel"] == "city"]
mean_per_month = df_city["IsCanceled"].groupby(pd.to_datetime(df_city[["ArrivalDateYear","ArrivalDateMonth"]].astype(str).agg('-'.join, axis=1) + '-01')).mean()
plt.scatter(mean_per_month.index,mean_per_month.values)

In [ ]:
# df_city.loc[df_city["arrival_date"]<pd.to_datetime("2015-08-01"),"IsCanceled"].mean()
df_city.loc[np.logical_and(df_city["arrival_date"]<pd.to_datetime("2015-08-01"), df_city["booking_date"]<pd.to_datetime("2015-07-01")),"IsCanceled"].describe()

In [ ]:
# Most bookings are made close to the arrival date, but there are also some bookings with a very long lead time.
import matplotlib.pyplot as plt
_ = plt.hist(df["LeadTime"], bins=1000)

In [ ]:
# # Inspect duplicates
# duplicates = df.loc[df.duplicated(keep=False)]
# row_key = duplicates.astype(str).agg('|'.join, axis=1)
# row_hash = row_key.map(hash)   # okay for quick use, but not stable across runs

# duplicates_sorted = duplicates.assign(_hash=row_hash).sort_values('_hash')
# duplicates_sorted.head(7).T
# Decision: Drop all duplicates, as it is nearly impossible that so many bookings have are booked the same amouint of days in advance on the exact same date for the exact same date. As the paper mentioned that the data was joined from multiple databases, it can be assumed that this is a data colelction artifact, rather than systematic.

In [ ]:
# Find out how many days in advance the status was changed
status_change_prior_booking = (df["arrival_date"]-pd.to_datetime(df["ReservationStatusDate"])).dt.days

# Whenever the last status change was after the arrival date, the booking was not cancelled. 
df.loc[status_change_prior_booking<0, "IsCanceled"].value_counts()

In [ ]:
df.loc[status_change_prior_booking>0, "IsCanceled"].value_counts()

In [ ]:
_ = plt.hist(status_change_prior_booking[status_change_prior_booking>0], bins=400)


In [ ]:
# need to drop reservation status
pd.crosstab(df["ReservationStatus"], df["IsCanceled"])

In [ ]:
test_df[["booking_date", "arrival_date", "ReservationStatusDate"]].describe()

In [ ]:
train_df[["booking_date", "arrival_date", "ReservationStatusDate"]].describe()

## Report

`ds.check()` runs the full pipeline and the bundle checks without saving and writes `README.md`. `ds.build()` also
saves the container (new UUID); only the curator runs it.

In [ ]:
result = ds.check()